# Auditoría de sesgo por grupo

El modelo usa `GENERO`, `RANGO_SALARIO`, `ZONA_RESIDENCIA` y `RANGO_EDAD` como predictores, de
modo que puede tener tasas de error distintas entre grupos. Este notebook lo mide.

## Qué métrica importa aquí

Para un sistema que **prioriza a quién contactar**, la métrica relevante es la **paridad de
recall**, también llamada *igualdad de oportunidad*: un estudiante en riesgo debería tener la
misma probabilidad de ser detectado sea cual sea su grupo. Si el recall difiere, hay grupos que
reciben menos acompañamiento con el mismo nivel de riesgo.

La paridad de *precisión* importa menos: un falso positivo cuesta una llamada. La paridad
demográfica —seleccionar la misma proporción de cada grupo— sería incorrecta como objetivo,
porque las tasas base de deserción difieren realmente entre grupos.

Se aplica la **regla del 80 %**: si el recall del peor grupo es menos del 80 % del recall del
mejor, hay disparidad que exige explicación.

## Resultado

| Atributo | Razón de recall | Estado |
|---|---|---|
| `GENERO` | 0.921 | aceptable |
| `MODALIDAD` | 0.499 | **disparidad** |
| `RANGO_SALARIO` | 0.503 | **disparidad** |
| `ZONA_RESIDENCIA` | 0.374 | **disparidad** |
| `RANGO_EDAD` | **0.099** | **disparidad severa** |

Y un hallazgo que atraviesa a todos los atributos, en la última sección.

## Preparación

In [ ]:
import os, warnings
import numpy as np, pandas as pd
from sqlalchemy import create_engine, text
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')

host, port, database = os.getenv('DB_HOST'), os.getenv('DB_PORT', '1433'), os.getenv('DB_NAME')
VALOR_DESERCION = os.getenv('STATUS_DESERCION', '').strip().lower()
if not all([host, database, VALOR_DESERCION]):
    raise RuntimeError("Define DB_HOST, DB_NAME y STATUS_DESERCION. Ver .env.example")
engine = create_engine(
    'mssql+pyodbc://@{h}:{p}/{d}?trusted_connection=yes'
    '&driver=ODBC+Driver+18+for+SQL+Server&TrustServerCertificate=yes'.format(
        h=host, p=port, d=database), connect_args={'timeout': 300})

def consultar(sql):
    with engine.connect() as cn:
        return pd.read_sql_query(text(sql), cn)

In [ ]:
base = consultar('''
    SELECT Identificacion, Periodo, Status, Tipo_Salto, Modalidad, Semestre_SINU,
           año AS ANIO, Genero, RANGO_EDAD, RANGO_SALARIO, ESTA_TRABAJANDO,
           METODO_FINANCIAMIENTO, ZONA_RESIDENCIA, REGIMEN_SISTEMA_SALUD
    FROM academico.historial_academico''')
n0 = len(base)
for sql in ['''SELECT IDENTIFICACION AS Identificacion, COD_PERIODO AS Periodo,
                      [MATERIAS INSCRITAS] AS MATERIAS_INSCRITAS,
                      [MATERIAS APROBADAS] AS MATERIAS_APROBADAS,
                      Porcentaje_aprobacion AS PORCENTAJE_APROBACION
               FROM academico.aprobacion_materias''',
            'SELECT Identificacion, Periodo, TOTAL FROM financiera.cartera']:
    aux = consultar(sql).drop_duplicates(subset=['Identificacion', 'Periodo'], keep='last')
    base = base.merge(aux, on=['Identificacion', 'Periodo'], how='left')
assert len(base) == n0

base.columns = [c.strip().upper() for c in base.columns]
base['TARGET'] = (base['STATUS'].astype(str).str.strip().str.lower() == VALOR_DESERCION).astype(int)
base['FLAG_NO_APROBO_NADA'] = np.where(base['PORCENTAJE_APROBACION'] == 0, 1, 0)
base['PORCENTAJE_APROBACION'] = base['PORCENTAJE_APROBACION'].replace(0, np.nan)
base['ANIO'] = pd.to_numeric(base['ANIO'], errors='coerce')
base = base.dropna(subset=['ANIO']); base['ANIO'] = base['ANIO'].astype(int)

NUM = ['SEMESTRE_SINU', 'MATERIAS_INSCRITAS', 'MATERIAS_APROBADAS',
       'PORCENTAJE_APROBACION', 'TOTAL', 'FLAG_NO_APROBO_NADA']
CAT = ['TIPO_SALTO', 'MODALIDAD', 'GENERO', 'RANGO_EDAD', 'RANGO_SALARIO',
       'ESTA_TRABAJANDO', 'METODO_FINANCIAMIENTO', 'ZONA_RESIDENCIA',
       'REGIMEN_SISTEMA_SALUD']
F = NUM + CAT
for c in NUM: base[c] = pd.to_numeric(base[c], errors='coerce')
for c in CAT: base[c] = base[c].astype(str)

MARGEN_MADURACION = 2
ANIO_TEST = int(base['ANIO'].max()) - MARGEN_MADURACION
tr = base[base['ANIO'] == ANIO_TEST - 1]
te = base[base['ANIO'] == ANIO_TEST].copy()
spw = (tr['TARGET'] == 0).sum() / (tr['TARGET'] == 1).sum()

modelo = Pipeline([
    ('prep', ColumnTransformer([
        ('n', Pipeline([('i', SimpleImputer(strategy='median')),
                        ('s', StandardScaler())]), NUM),
        ('c', Pipeline([('i', SimpleImputer(strategy='most_frequent')),
                        ('o', OrdinalEncoder(handle_unknown='use_encoded_value',
                                             unknown_value=-1))]), CAT)])),
    ('clf', LGBMClassifier(n_estimators=481, num_leaves=32, learning_rate=0.0244,
                           subsample=0.9122, random_state=42, n_jobs=-1,
                           verbose=-1, scale_pos_weight=spw))])
modelo.fit(tr[F], tr['TARGET'])
te['PROBA'] = modelo.predict_proba(te[F])[:, 1]

# Punto de operación: los del top 10 % de riesgo, como haría el programa
CAPACIDAD = 0.10
corte = np.sort(te['PROBA'])[::-1][int(len(te) * CAPACIDAD) - 1]
te['SELECCIONADO'] = te['PROBA'] >= corte

print(f"Punto de operación: top {CAPACIDAD:.0%}")
print(f"Global -> AUC {roc_auc_score(te['TARGET'], te['PROBA']):.4f} | "
      f"recall {te.loc[te['TARGET']==1,'SELECCIONADO'].mean():.4f} | "
      f"precision {te.loc[te['SELECCIONADO'],'TARGET'].mean():.4f}")

## Desempeño desagregado

Se reportan **proporciones**, no conteos: los volúmenes por grupo son información
institucional. Se omiten los grupos con menos de 500 registros, donde la métrica sería ruido.

In [ ]:
def auditar(atributo, minimo=500):
    filas = []
    for g, sub in te.groupby(atributo):
        if len(sub) < minimo or sub['TARGET'].nunique() < 2:
            continue
        filas.append({
            'grupo': str(g)[:30],
            'peso_%': round(len(sub) / len(te) * 100, 1),
            'tasa_base': round(float(sub['TARGET'].mean()), 4),
            'tasa_seleccion': round(float(sub['SELECCIONADO'].mean()), 4),
            'recall': round(float(sub.loc[sub['TARGET'] == 1, 'SELECCIONADO'].mean()), 4),
            'precision': round(float(sub.loc[sub['SELECCIONADO'], 'TARGET'].mean()), 4),
            'auc': round(float(roc_auc_score(sub['TARGET'], sub['PROBA'])), 4),
        })
    t = pd.DataFrame(filas).sort_values('recall', ascending=False)
    if len(t) > 1:
        razon = t['recall'].min() / t['recall'].max()
        estado = 'DISPARIDAD' if razon < 0.8 else 'dentro de la regla del 80 %'
        print(f"{atributo}: razón de recall {razon:.3f} -> {estado}")
    return t


for a in ['GENERO', 'MODALIDAD', 'RANGO_SALARIO', 'ZONA_RESIDENCIA', 'RANGO_EDAD']:
    print()
    display(auditar(a))

## El hallazgo que atraviesa todos los atributos

En **cada** atributo, el grupo con mayor recall es el de valor ausente (`None`). Con el punto
de operación en el top 10 %:

| Atributo | Recall del grupo sin dato | Recall del mejor grupo con dato |
|---|---|---|
| `RANGO_EDAD` | 0.701 | 0.162 |
| `ZONA_RESIDENCIA` | 0.523 | 0.199 |
| `RANGO_SALARIO` | 0.320 | 0.213 |

**El modelo está detectando sobre todo a estudiantes con ficha incompleta.** Y no es un
artefacto de codificación: quienes tienen datos faltantes desertan más —tasa base 0.272 frente
a 0.16–0.19 en los grupos con edad declarada— así que el patrón es real y el modelo lo explota
correctamente.

El problema es lo que implica: **para la mayoría de estudiantes, que sí tienen ficha completa,
el modelo funciona bastante peor de lo que sugiere su métrica global.** El desempeño agregado
se apoya en un segmento donde la señal es fácil.

### La disparidad más preocupante

Con el grupo sin dato apartado, la brecha por edad sigue siendo grande y va en la peor
dirección posible: los estudiantes de **16 a 20 años** obtienen un recall de **0.069**, frente
a 0.16 en los tramos de mayor edad, pese a tener una tasa base de deserción similar (0.175).

Su precisión es la más alta de todos los grupos (0.798), lo que confirma el mecanismo: el
modelo solo los señala cuando está muy seguro, y por eso se le escapan casi todos. Son
justamente los estudiantes de primer ingreso, sobre los que un programa de permanencia tiene
más margen de acción.

---
## Mitigación: repartir la capacidad por segmento

La disparidad no viene de que el modelo sea injusto por diseño, sino de cómo se usa: un corte
global asigna menos acompañamiento a los grupos que el modelo predice peor, justamente por ser
más difíciles de predecir. Repartir la misma proporción **dentro de cada grupo** rompe esa
relación.

Se comparan tres formas de repartir la misma capacidad total.

In [ ]:
CAPACIDAD = 0.10
K = int(len(te) * CAPACIDAD)
ATRIBUTO = 'RANGO_EDAD'


def evaluar(seleccion, etiqueta):
    filas = []
    for g, sub in te.groupby(ATRIBUTO):
        if len(sub) < 500 or sub['TARGET'].nunique() < 2:
            continue
        s = seleccion[sub.index]
        filas.append({'grupo': str(g)[:26],
                      'seleccion': round(float(s.mean()), 4),
                      'recall': round(float(s[sub['TARGET'] == 1].mean()), 4)})
    t = pd.DataFrame(filas).sort_values('recall', ascending=False)
    razon = t['recall'].min() / t['recall'].max()
    print(f"{etiqueta}")
    print(f"  paridad {razon:.3f} | desertores captados {int(te.loc[seleccion,'TARGET'].sum()):,} "
          f"| precision {te.loc[seleccion,'TARGET'].mean():.4f}")
    return t, razon


# A ─ global: los K de mayor riesgo, sin mirar grupo
sel_a = pd.Series(False, index=te.index)
sel_a[te['PROBA'].nlargest(K).index] = True
tabla_a, _ = evaluar(sel_a, 'A  capacidad global (lo actual)')

# B ─ misma proporcion dentro de cada grupo
sel_b = pd.Series(False, index=te.index)
for g, sub in te.groupby(ATRIBUTO):
    k = int(round(len(sub) * CAPACIDAD))
    if k:
        sel_b[sub['PROBA'].nlargest(k).index] = True
tabla_b, _ = evaluar(sel_b, 'B  misma tasa por grupo')

# C ─ capacidad proporcional al riesgo esperado de cada grupo
sel_c = pd.Series(False, index=te.index)
cuota = (te.groupby(ATRIBUTO)['PROBA'].sum() / te['PROBA'].sum() * K).round().astype(int)
for g, sub in te.groupby(ATRIBUTO):
    k = min(int(cuota.get(g, 0)), len(sub))
    if k:
        sel_c[sub['PROBA'].nlargest(k).index] = True
tabla_c, _ = evaluar(sel_c, 'C  proporcional al riesgo del grupo')

display(tabla_b)

**Resultado.**

| Estrategia | Paridad | Desertores captados | Precisión |
|---|---|---|---|
| A global *(lo actual)* | **0.099** | 6.815 | 0.597 |
| **B misma tasa por grupo** | **0.850** | 6.564 *(−3,7 %)* | 0.575 |
| C proporcional al riesgo | 0.657 | 6.768 *(−0,7 %)* | 0.593 |

**La estrategia B cierra la brecha**: lleva la paridad de 0.099 a 0.850, por encima de la regla
del 80 %. Para el grupo de 16 a 20 años el recall pasa de **0.069 a 0.313**: cuatro veces y
media más detección.

El costo es 251 desertores menos captados sobre 6.815 y dos puntos de precisión. A cambio, el
acompañamiento deja de depender de qué tan fácil es predecir a cada grupo.

Conviene notar que **C, la opción teóricamente más elegante** —repartir en proporción al riesgo
esperado de cada grupo— **no funciona**: se queda en 0.657, porque la masa de riesgo del grupo
sin datos absorbe la cuota. La solución simple gana a la sofisticada.

`priorizar()` en [`Prototipo 1.8.ipynb`](Prototipo%201.8.ipynb) implementa B por defecto, y
avisa por consola si se le pide un reparto global.

### Lo que sigue pendiente

1. **Completar la ficha sociodemográfica.** Buena parte del desempeño actual proviene de la
   ausencia de datos, no de entender el fenómeno. La mitigación reparte mejor, pero no mejora
   la señal.
2. **Reauditar tras cada reentrenamiento.** Esta medición corresponde a una extracción concreta
   y las fuentes de este proyecto cambian.